# DED - Sachsen

In [1]:
import warnings

from tqdm import tqdm
import zipfile
import pandas as pd
import pyproj
from camelsp import get_metadata

from camels_de1h import get_input_path, get_nuts_id_from_provider_id, Station1h

## Extract data

In [2]:
input_path = get_input_path("DED")

# extract zip
if not (input_path / "Q_und_W").exists():
    with zipfile.ZipFile(input_path / "Sachsen_Q_und_W.zip", "r") as zip_ref:
        zip_ref.extractall(input_path)
        print("Data extracted.")
else:
    print("Data already extracted.")

Data already extracted.


## Parse data


In [3]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    raw_meta_all = pd.read_excel(input_path / "Stammdaten_Sachsen.xlsx")

raw_meta_all["Pegelkennziffer"] = raw_meta_all["Pegelkennziffer"].astype(str)

ids = raw_meta_all["Pegelkennziffer"].values

In [4]:
ids_files = [str(f).split("_")[-1].split(".")[0] for f in (input_path / "Q_und_W").iterdir() if f.is_file()]

files_without_metadata = []

# check if all ids_files are in ids
for id_file in ids_files:
    if id_file not in ids:
        files_without_metadata.append(id_file)

if len(files_without_metadata) > 0:
    print("Files without metadata:")
    print(files_without_metadata)
else:
    print("All files / stations have metadata.")

All files / stations have metadata.


In [5]:
def parse_station_data(id: str, aggregate_hourly: bool) -> pd.DataFrame:
    # get file name
    fname = [f for f in input_path.glob(f"Q_und_W/*{id}.xlsx")][0]

    # read data
    with warnings.catch_warnings():
    # ignore UserWarning about default style of openpyxl
        warnings.filterwarnings("ignore", category=UserWarning)
        data = pd.read_excel(fname, skiprows=2, parse_dates=True, date_format="%Y-%m-%d %H:%M:%S")

    data = data[["Zeitpunkt (MEZ)", "Wasserstand (W) cm", "Durchfluss (Q) m³/s"]]
    data.columns = ["date", "water_level_obs", "discharge_vol_obs"]

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    return data

In [6]:
def check_metadata_consistency(camelsp_meta, raw_meta, id: str) -> dict:
    """
    Check consistency of metadata fields between raw and camelsp sources.
    """
    inconsistencies = {}
    
    fields = {
        "provider_id": {
            "camelsp": "provider_id",
            "raw": "Pegelkennziffer"
        },
        "gauge_name": {
            "camelsp": "gauge_name",
            "raw": "Pegelname"
        },
        "waterbody": {
            "camelsp": "waterbody_name",
            "raw": "Gewaesser"
        },
        "area": {
            "camelsp": "area",
            "raw": "Fläche Einzugsgebiet km²"
        },
    }

    for field_key, field_info in fields.items():
        raw_val = raw_meta[field_info["raw"]].values[0]
        camelsp_val = camelsp_meta[field_info["camelsp"]].values[0]
        
        if raw_val != camelsp_val:
            if id in inconsistencies:
                inconsistencies[id].update({field_key: {
                    "raw": raw_val,
                    "camelsp": camelsp_val
                }})
            else:
                inconsistencies[id] = {field_key: {
                    "raw": raw_val,
                    "camelsp": camelsp_val
                }}
    
    return inconsistencies

In [ ]:
camelsp_meta_all = get_metadata()

metadata_inconsistencies = {}

not_in_camelsp = []

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    try:
        gauge_id = get_nuts_id_from_provider_id(id, "DED", add_missing=False)
    except ValueError:
        not_in_camelsp.append(id)
        continue

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    raw_meta = raw_meta_all[raw_meta_all["Pegelkennziffer"] == id]

    # check the consistency of the metadata
    try:
        inconsistencies = check_metadata_consistency(camelsp_meta, raw_meta, id)
    except IndexError:
        print(f"Error for station {id}")
        continue
    if inconsistencies:
        metadata_inconsistencies.update(inconsistencies)

metadata_inconsistencies

# make it a panads DataFrame with the station_id as index
pd.DataFrame(metadata_inconsistencies).T

  0%|          | 0/183 [00:00<?, ?it/s]

 13%|█▎        | 24/183 [00:00<00:00, 336.73it/s]

Error for station 576420


""


Just different information about catchment sizes for two stations in CAMELS-DE daily and the raw metadata. I think we can ignore this and just use the CAMELS-DE metadata.

In CAMELS-DE daily, we do not have any information from the federal states about gauge elevation. So we always use the new `raw_metadata` here (and possibly also update the CAMELS-DE daily metadata in the future).

In [7]:
# get the metadata from CAMELS-DE daily
camelsp_meta_all = get_metadata()

ids = [str(f).split("_")[-1].split(".")[0] for f in (input_path / "Q_und_W").glob("*.xlsx")]

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DED", add_missing=True)

    # parse data for the station
    data = parse_station_data(id, aggregate_hourly=True)

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    raw_meta = raw_meta_all[raw_meta_all["Pegelkennziffer"] == id]

    if not camelsp_meta.empty:
        # gauge elevation from raw metadata
        if not raw_meta.empty:
            gauge_elev = raw_meta["Pegelnullpunkt (m)"].values[0]

        # set metadata values from camelsp metadata
        station_meta = {
            "gauge_id": camelsp_meta["camels_id"].values[0],
            "provider_id": camelsp_meta["provider_id"].values[0],
            "gauge_name": camelsp_meta["gauge_name"].values[0],
            "waterbody_name": camelsp_meta["waterbody_name"].values[0],
            "federal_state": camelsp_meta["federal_state"].values[0],
            "gauge_lon": camelsp_meta["lon"].values[0],
            "gauge_lat": camelsp_meta["lat"].values[0],
            "gauge_easting": camelsp_meta["x"].values[0],
            "gauge_northing": camelsp_meta["y"].values[0],
            "gauge_elev_metadata": gauge_elev,
            "area_metadata": camelsp_meta["area"].values[0],
            "part_of_camelsp": True,
        }
    elif not raw_meta.empty:
        # parse the location
        easting, northing = raw_meta["Ostwert"].values[0], raw_meta["Nordwert"].values[0]

        # from EPSG:25832 to EPSG:4326
        transformer = pyproj.Transformer.from_crs("epsg:32633", "epsg:4326", always_xy=True)
        lon, lat = transformer.transform(easting, northing)

        # from EPSG:31466 to EPSG:3035
        transformer = pyproj.Transformer.from_crs("epsg:32633", "epsg:3035", always_xy=True)
        easting, northing = transformer.transform(easting, northing)

        # if the station is not in the camelsp metadata, use the raw metadata
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": raw_meta["Pegelname"].values[0],
            "waterbody_name": raw_meta["Gewaesser"].values[0],
            "federal_state": "Sachsen",
            "gauge_lon": lon,
            "gauge_lat": lat,
            "gauge_easting": easting,
            "gauge_northing": northing,
            "gauge_elev_metadata": raw_meta["Pegelnullpunkt (m)"].values[0],
            "area_metadata": raw_meta["Fläche Einzugsgebiet km²"].values[0],
            "part_of_camelsp": False,
        }
    else:
        raise ValueError(f"Station {id} has no metadata.")
    
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(raw_meta)
    

100%|██████████| 183/183 [2:19:00<00:00, 45.58s/it]  
